In [15]:
from collections import Counter
from typing import Union

import pandas as pd
import numpy as np
import sklearn as sk
import random
import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


ds = load_dataset("sh0416/ag_news")
train_df = pd.DataFrame(ds["train"])
test_df = pd.DataFrame(ds["test"])


In [14]:
print("Example Training Entry:")
print(train_df.head(1))
print("\nDistribution of Training Labels:")
print(train_df.groupby("label").size())

print("\nExample Testing Entry:")
print(test_df.head(1))
print("\nDistribution of Testing Labels:")
print(test_df.groupby("label").size())

Example Training Entry:
   label                                              title  \
0      3  Wall St. Bears Claw Back Into the Black (Reuters)   

                                         description  
0  Reuters - Short-sellers, Wall Street's dwindli...  

Distribution of Training Labels:
label
1    30000
2    30000
3    30000
4    30000
dtype: int64

Example Testing Entry:
   label                              title  \
0      3  Fears for T N pension after talks   

                                         description  
0  Unions representing workers at Turner   Newall...  

Distribution of Testing Labels:
label
1    1900
2    1900
3    1900
4    1900
dtype: int64


In [27]:
total_train_np = train_df.to_numpy()
train, val = train_test_split(
    total_train_np,
    test_size=0.1,
    random_state=SEED,
)
test = test_df.to_numpy()
print(train.shape, val.shape, test.shape)

train_examples = train[:, 1:]
train_labels = train[:, 0]
val_examples = val[:, 1:]
val_labels = val[:, 0]
test_examples = test[:, 1:]
test_labels = test[:, 0]

print("Num Training Examples:", len(train))
print("Num Validation Examples:", len(val))
print("Num Testing Examples:", len(test))


(108000, 3) (12000, 3) (7600, 3)
Num Training Examples: 108000
Num Validation Examples: 12000
Num Testing Examples: 7600


In [28]:
from torch.utils.data import dataset
from typing import Union

# should already be tokenized
class NewsDataset(dataset.Dataset):
    def __init__(self, examples: Union[torch.Tensor, np.ndarray], labels: Union[torch.Tensor, np.ndarray]):
        self.examples = examples if isinstance(examples, torch.Tensor) else torch.Tensor(examples)
        self.labels = labels if isinstance(labels, torch.Tensor) else torch.Tensor(labels)

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx], self.labels[idx]

In [29]:
import re

class Tokenizer:
    def __init__(self):
        # token to index (basically same as one hot
        self.vocab = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.index = 4
        self.initialized = False

    def _entry_to_tokens(self, entry):
        text = entry[0] + " " + entry[1]
        text = str(text).lower()
        tokens = re.findall(r"[a-z0-9']+", text)
        return tokens

    def build_vocab(self, examples: Union[torch.Tensor, np.ndarray]):
        if isinstance(examples, torch.Tensor):
            examples = examples.numpy()

        counts = Counter()

        # each example has title, desc
        for example in examples:
            # join title and description, lowercase, and extract tokens
            tokens = self._entry_to_tokens(example)
            counts.update(tokens)

        for token, freq in counts.items():
            if freq > 1 and token not in self.vocab:
                self.vocab[token] = self.index
                self.index += 1

        self.initialized = True

    def lookup(self, word: str) -> int:
        if word not in self.vocab:
            return self.vocab["<UNK>"]
        return self.vocab[word]

    def tokenize_entries(self, entries: Union[torch.Tensor, np.ndarray], max_len = 128):
        if isinstance(entries, torch.Tensor):
            entries = entries.numpy()
        sequences = np.full_like((len(entries), max_len), self.vocab["<PAD>"], dtype="int32")
        sequences[:, 0] = self.vocab["<SOS>"]
        for i, entry in enumerate(entries):
            tokens = self._entry_to_tokens(entry)
            for j, token in enumerate(tokens[:max_len-2]):
                sequences[i, j+1] = self.lookup(token)
            sequences[i, min(len(tokens)+1, max_len-1)] = self.vocab["<EOS>"]
        return sequences


